# Panel de viaje · Fase 1.4 · Puntos de interés por pueblo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tragabytes/panel-viaje/blob/main/tests/fase1_pois.ipynb)

**Proyecto:** https://github.com/tragabytes/panel-viaje  
**Sesión:** 05 — evaluación de Wikidata SPARQL, Wikipedia REST y Overpass como fuentes de POIs por municipio

## Qué evalúa este notebook

1. **Wikidata SPARQL** como fuente estructurada de POIs (monumentos + naturaleza) por municipio.
2. **Wikipedia REST API** + **Wikidata image** (cascada multi-fuente) para extracts y miniaturas.
3. **Overpass API** como fallback geográfico cuando Wikidata no devuelve suficiente.
4. **Wikidata SPARQL** (segunda consulta) para datos del municipio. Cierra el bloque 1.5 del plan.
5. **Overpass** para encontrar los pueblos más cercanos a un punto GPS. Crítico: alimenta la vista 3 entera.

## Muestra de pueblos

| Municipio | Provincia | Habitantes aprox | Por qué está en la muestra |
|---|---|---|---|
| Medinaceli | Soria | ~700 | Pueblo pequeño con historia romana. Caso del plan. Esperable: arco romano, castillo, colegiata. |
| Albarracín | Teruel | ~1000 | Pueblo pequeño con mucho turismo. Esperable: cobertura alta. |
| Chinchón | Madrid | ~5000 | Mediano con patrimonio reconocible (plaza mayor, castillo). |
| Trujillo | Cáceres | ~9000 | Mediano con mucho patrimonio (castillo, Pizarro, iglesias). |
| Pedraza | Segovia | ~400 | Caso límite: muy pequeño pero municipio (no pedanía). Conjunto Histórico-Artístico desde 1951. |

## Tipos de POI que buscamos

Patrimonio histórico + naturaleza: castillos, iglesias, monasterios, palacios, puentes históricos, yacimientos arqueológicos, arcos romanos, murallas, torres, miradores, picos, espacios naturales. Sin museos, sin plazas, sin bares.

## Estado al cierre de la sesión 05

- ✅ Prueba 1: Wikidata SPARQL (POIs) — cerrada y aprobada tras varias iteraciones de filtrado.
- ✅ Prueba 2: Wikipedia REST + Wikidata image (cascada) — cerrada y aprobada.
- ⏸️ Prueba 3: Overpass como fallback — pendiente. Tres mirrors saturados durante la sesión.
- ✅ Prueba 4: Wikidata SPARQL (datos del municipio) — cerrada. Absorbe el bloque 1.5 del plan.
- ⏸️ Prueba 5: Overpass para pueblos cercanos — pendiente, mismo motivo que la 3.

## Cómo ejecutar

- Abre este notebook con el badge `Open in Colab` de arriba o desde el README.
- Ejecuta las celdas de arriba abajo. Cada prueba es independiente y se puede ejecutar suelta tras el setup.
- Las pruebas 3 y 5 dependen de Overpass; si falla, salta a las pruebas 4 y al resumen.
- Al final hay un resumen que se puede copiar al documento de seguimiento.

## Setup: imports, constantes y helpers

In [ ]:
import requests
import json
import time
import random
from urllib.parse import quote

# User-Agent propio, obligatorio para Wikidata/Wikipedia y buena ciudadanía en Overpass.
# Quedó como estándar del proyecto desde la sesión 03 (problema P02 con Photon).
UA = "panel-viaje-tragabytes/0.1 (https://github.com/tragabytes/panel-viaje; contacto via GitHub)"
HEADERS = {"User-Agent": UA, "Accept": "application/json"}

# Endpoints
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WIKIPEDIA_REST_ES = "https://es.wikipedia.org/api/rest_v1"
# Overpass: el endpoint principal y los mirrors conocidos. Si uno está saturado,
# probar con el siguiente. En la sesión 05 los tres estaban caídos al mismo tiempo
# (saturación global, no problema del notebook).
OVERPASS = "https://overpass-api.de/api/interpreter"
# Mirrors alternativos:
# OVERPASS = "https://overpass.kumi.systems/api/interpreter"
# OVERPASS = "https://overpass.private.coffee/api/interpreter"

# Los 5 municipios de la muestra. NO incluimos QIDs aquí: los resolveremos
# automáticamente en la siguiente celda preguntando a Wikidata por nombre + provincia.
# Esto evita errores de QIDs metidos a mano (problema P04 de la sesión 05).
MUNICIPIOS_BASE = [
    {"nombre": "Medinaceli",                  "provincia": "Soria",   "lat": 41.1731, "lon": -2.4344, "habs_aprox": 700,  "caso": "pequeño con historia"},
    {"nombre": "Albarracín",                  "provincia": "Teruel",  "lat": 40.4064, "lon": -1.4433, "habs_aprox": 1000, "caso": "pequeño turístico"},
    {"nombre": "Chinchón",                    "provincia": "Madrid",  "lat": 40.1396, "lon": -3.4261, "habs_aprox": 5000, "caso": "mediano"},
    {"nombre": "Trujillo",                    "provincia": "Cáceres", "lat": 39.4600, "lon": -5.8825, "habs_aprox": 9000, "caso": "mediano con mucho patrimonio"},
    {"nombre": "Pedraza",                     "provincia": "Segovia", "lat": 41.1283, "lon": -3.8108, "habs_aprox": 400,  "caso": "muy pequeño, caso límite"},
]

# Patrón de reintentos estándar del proyecto (validado en sesión 04 con Open-Meteo).
# Timeout 30 s, hasta 3 intentos, esperas crecientes 3 s y 8 s.
def fetch_json(url, method="GET", data=None, headers=None, max_intentos=3, timeout=30):
    hdrs = headers or HEADERS
    esperas = [0, 3, 8]
    last_err = None
    for intento in range(max_intentos):
        if esperas[intento] > 0:
            time.sleep(esperas[intento])
        t0 = time.time()
        try:
            if method == "POST":
                r = requests.post(url, data=data, headers=hdrs, timeout=timeout)
            else:
                r = requests.get(url, headers=hdrs, timeout=timeout)
            dt = (time.time() - t0) * 1000
            if r.status_code == 200:
                return {"ok": True, "data": r.json(), "ms": dt, "bytes": len(r.content), "intentos": intento + 1}
            last_err = f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
    return {"ok": False, "error": last_err, "intentos": max_intentos}

# Versión específica para Overpass: timeout más largo, reintentos más espaciados.
# Overpass es notoriamente inestable y puede dar 504/429 transitorios incluso en
# servidores supuestamente fiables. Validado en sesión 05.
def fetch_overpass(query, max_intentos=4, timeout=60):
    esperas = [0, 5, 15, 30]
    last_err = None
    for intento in range(max_intentos):
        if esperas[intento] > 0:
            time.sleep(esperas[intento])
        t0 = time.time()
        try:
            r = requests.post(OVERPASS, data={"data": query},
                              headers={"User-Agent": UA}, timeout=timeout)
            dt = (time.time() - t0) * 1000
            if r.status_code == 200:
                return {"ok": True, "data": r.json(), "ms": dt, "bytes": len(r.content), "intentos": intento + 1}
            last_err = f"HTTP {r.status_code}: {r.text[:150]}"
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
    return {"ok": False, "error": last_err, "intentos": max_intentos}

print("Setup OK")
print(f"Muestra base: {len(MUNICIPIOS_BASE)} municipios (los QIDs se resuelven en la celda siguiente)")
for m in MUNICIPIOS_BASE:
    print(f"  {m['nombre']:32s} ({m['provincia']})  [{m['caso']}]")

## Resolución automática de QIDs de Wikidata

**Por qué esta celda existe:** los identificadores externos como QIDs de Wikidata, IDs de OSM o códigos INE no pueden meterse a mano. En el primer intento de la sesión 05, los QIDs metidos por conocimiento general apuntaban a entidades incorrectas en 5/5 casos (Medinaceli a una cordillera de Oregón, Albarracín a un sitio en Cataluña, etc.). Aprendizaje del proyecto: los identificadores se obtienen siempre por consulta verificable.

**Cómo se resuelve:** para cada municipio, una consulta SPARQL busca un item con esa label en español, instancia (o subclase transitiva) de Q2074737 (municipio de España), y filtra por provincia para deshacer ambigüedades. Imprime los QIDs resueltos con enlaces a wikidata.org para verificación visual.

In [ ]:
def resolver_qid_municipio(nombre, provincia):
    """Busca el QID de un municipio español por nombre + provincia."""
    query = f'''
SELECT DISTINCT ?item ?itemLabel ?provLabel WHERE {{
  ?item wdt:P31/wdt:P279* wd:Q2074737 .
  ?item rdfs:label "{nombre}"@es .
  OPTIONAL {{ ?item wdt:P131 ?prov. }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es". }}
}}
LIMIT 20
'''
    url = f"{WIKIDATA_SPARQL}?format=json&query={quote(query)}"
    hdrs = {"User-Agent": UA, "Accept": "application/sparql-results+json"}
    res = fetch_json(url, headers=hdrs)
    if not res['ok']:
        return None, f"error: {res['error'][:80]}"
    rows = res['data']['results']['bindings']
    if not rows:
        return None, "sin resultados"
    candidatos = []
    for row in rows:
        qid = row['item']['value'].rsplit('/', 1)[-1]
        prov = row.get('provLabel', {}).get('value', '')
        candidatos.append((qid, prov))
    def norm(s): return s.lower().replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')
    for qid, prov in candidatos:
        if norm(provincia) in norm(prov):
            return qid, f"match por provincia ({prov})"
    qid, prov = candidatos[0]
    return qid, f"primer match (provincia devuelta: {prov or '?'}) - VERIFICAR"

MUNICIPIOS = []
for m in MUNICIPIOS_BASE:
    qid, motivo = resolver_qid_municipio(m['nombre'], m['provincia'])
    if qid:
        m_copia = dict(m)
        m_copia['qid'] = qid
        MUNICIPIOS.append(m_copia)
        print(f"  ✓ {m['nombre']:32s} -> {qid:12s}  [{motivo}]")
    else:
        print(f"  ✗ {m['nombre']:32s} NO RESUELTO  [{motivo}]")
    time.sleep(1)

print(f"\nTotal resuelto: {len(MUNICIPIOS)}/{len(MUNICIPIOS_BASE)}")
print("\nVerificación visual: abre cada QID en wikidata.org/wiki/QID y confirma que es el municipio correcto.")
for m in MUNICIPIOS:
    print(f"  https://www.wikidata.org/wiki/{m['qid']}  ({m['nombre']})")

## Prueba 1 — Wikidata SPARQL: POIs por municipio

**Estrategia (versión final tras 5 iteraciones en sesión 05):**

1. Para cada municipio, lanzamos una consulta SPARQL que busca elementos `wdt:P131*` (located in administrative entity, transitivamente) dentro del Q del municipio.
2. Filtrados por instancia (o subclase) de una **lista blanca** de clases de POI legítimas (castillos, iglesias, palacios, conjuntos históricos, miradores, etc.).
3. Excluidos por una **lista negra de instancias DIRECTAS** (no transitiva). Esto es importante: filtrar por subclase transitiva eliminaba castillos legítimos por herencia accidental de "monumento" en Wikidata. Filtrar por instancia directa caza vértices y fosas sin tocar lo bueno.
4. Filtros adicionales en Python tras recibir la respuesta: descarte por palabras clave en label (`stolperstein`) y deduplicación por contención de label normalizada (para eliminar duplicados como "Delimitación del Conjunto Histórico" vs "Conjunto Histórico de X").

**Aprendizaje meta importante:** los QIDs de las clases de exclusión también deben descubrirse por consulta, no de memoria. La sesión 05 reveló que `Q11691083` (mi suposición para vértice geodésico) era incorrecto; el real es `Q131862`. Mismo error con fosas comunes.

In [ ]:
# Cache buster aleatorio para evitar caché HTTP del proxy de Colab.
# Validado como necesario en sesión 05 (problema P07): sin esto, Colab cacheaba
# respuestas SPARQL idénticas dando latencias imposibles (70 ms) y resultados
# que no reflejaban los cambios de filtros.
def consultar_wikidata(query):
    cache_buster = random.randint(100000, 999999)
    url = f"{WIKIDATA_SPARQL}?format=json&_cb={cache_buster}&query={quote(query)}"
    hdrs = {"User-Agent": UA, "Accept": "application/sparql-results+json"}
    return fetch_json(url, headers=hdrs)

# Clases de Wikidata que consideramos POIs válidos para el panel.
CLASES_POI = [
    # Patrimonio religioso
    ("Q23413",    "castillo"),
    ("Q16970",    "iglesia"),
    ("Q2977",     "catedral"),
    ("Q108325",   "capilla"),
    ("Q44613",    "monasterio"),
    ("Q317557",   "iglesia parroquial"),
    ("Q56242063", "ermita"),
    ("Q1128397",  "convento"),
    # Patrimonio civil/militar
    ("Q16560",    "palacio"),
    ("Q12518",    "torre"),
    ("Q57821",    "fortificación"),
    ("Q1549591",  "edificio histórico"),
    ("Q1440300",  "monumento histórico"),
    ("Q4989906",  "monumento"),
    ("Q839954",   "yacimiento arqueológico"),
    ("Q1486652",  "puente de piedra"),
    ("Q1189170",  "arco triunfal"),
    ("Q2425052",  "puerta de la ciudad"),
    ("Q2319498",  "atalaya"),
    # Conjuntos histórico-artísticos (figura española importante)
    ("Q12015628", "Conjunto histórico-artístico"),
    # Naturaleza singular
    ("Q207326",   "mirador"),
    ("Q35509",    "cueva"),
    ("Q34038",    "cascada"),
    ("Q473972",   "área protegida"),
    ("Q46169",    "monumento natural"),
    ("Q179049",   "parque nacional"),
]

# Lista negra DIRECTA: si el item tiene EXACTAMENTE alguna de estas como wdt:P31, fuera.
# El filtro mira la instancia directa, no la cadena de subclases, para no eliminar
# castillos o iglesias por herencia accidental.
# Los QIDs de Q131862 y Q734271 fueron descubiertos por consulta de diagnóstico en
# sesión 05; las suposiciones iniciales (Q11691083, Q1241715) eran incorrectas.
CLASES_EXCLUIR_DIRECTAS = [
    "Q486972",     # asentamiento humano
    "Q2961800",    # entidad singular de población
    "Q15078955",   # núcleo de población
    "Q2074737",    # municipio de España (para que no se cuele el propio municipio)
    "Q3257686",    # localidad
    "Q39614",      # cementerio
    "Q7075",       # biblioteca
    "Q856584",     # biblioteca pública
    "Q131862",     # vértice geodésico (QID real, descubierto por diagnóstico)
    "Q3257430",    # condado jurisdiccional
    "Q1620908",    # ducado jurisdiccional
    "Q734271",     # fosa común (QID real, descubierto por diagnóstico)
    "Q314003",     # Stolperstein
    "Q15823516",   # adoquín conmemorativo
]

def construir_query_pois(qid_municipio):
    valores_in = " ".join(f"wd:{q}" for q, _ in CLASES_POI)
    valores_out = " ".join(f"wd:{q}" for q in CLASES_EXCLUIR_DIRECTAS)
    return f'''
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?instanceLabel ?image ?coord ?article WHERE {{
  ?item wdt:P131* wd:{qid_municipio} .
  ?item wdt:P31 ?instance .
  ?instance wdt:P279* ?parentClass .
  VALUES ?parentClass {{ {valores_in} }}
  # Exclusión por instancia DIRECTA (no transitiva): los vértices y fosas
  # no se cuelan aunque hereden de "monumento".
  FILTER NOT EXISTS {{
    ?item wdt:P31 ?instExcluida .
    VALUES ?instExcluida {{ {valores_out} }}
  }}
  OPTIONAL {{ ?item wdt:P18 ?image. }}
  OPTIONAL {{ ?item wdt:P625 ?coord. }}
  OPTIONAL {{
    ?article schema:about ?item ;
             schema:isPartOf <https://es.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es,en". }}
}}
LIMIT 30
'''

def score_relevancia(poi):
    """Score para ordenar: imagen + artículo es.wiki es lo más curado."""
    s = 0
    if poi.get('image'): s += 2
    if poi.get('article'): s += 2
    if poi.get('desc'): s += 1
    return -s  # negativo para que sort ascendente ponga lo mejor primero

def normalizar_label(label):
    """Normaliza para comparación: minúsculas, sin tildes, sin palabras de relleno."""
    import unicodedata, re
    s = unicodedata.normalize('NFD', label.lower())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'\b(delimitacion|del|de|la|los|las|el|y)\b', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def label_descarte(label):
    """True si la label contiene palabras que indican que NO es un POI turístico."""
    l = label.lower()
    descartes = ["stolperstein", "fosa común", "fosa comun", "vértice geodésico", "vertice geodesico"]
    return any(d in l for d in descartes)

def deduplicar_pois(pois):
    """Dedupa por contención de label normalizada: si una label contiene a la otra,
    se considera duplicado y se queda con el de mayor score."""
    pois_filtrados = [p for p in pois if not label_descarte(p['label'])]
    pois_ordenados = sorted(pois_filtrados, key=score_relevancia)
    resultado = []
    labels_norm = []
    for p in pois_ordenados:
        ln = normalizar_label(p['label'])
        if not ln:
            continue
        es_duplicado = False
        for ya in labels_norm:
            if ln in ya or ya in ln:
                es_duplicado = True
                break
        if not es_duplicado:
            resultado.append(p)
            labels_norm.append(ln)
    return resultado

# Lanzamos la consulta para cada municipio de la muestra.
resultados_wd = {}

for m in MUNICIPIOS:
    print(f"\n=== {m['nombre']} ({m['qid']}) ===")
    q = construir_query_pois(m['qid'])
    res = consultar_wikidata(q)
    if not res['ok']:
        print(f"  ERROR: {res['error']}")
        resultados_wd[m['nombre']] = {"ok": False, "error": res['error'], "pois": []}
        continue
    rows = res['data']['results']['bindings']
    pois_dict = {}
    for row in rows:
        uri = row['item']['value']
        if uri not in pois_dict:
            pois_dict[uri] = {
                "qid": uri.rsplit('/', 1)[-1],
                "label": row.get('itemLabel', {}).get('value', ''),
                "desc": row.get('itemDescription', {}).get('value', ''),
                "instances": set(),
                "image": row.get('image', {}).get('value'),
                "coord": row.get('coord', {}).get('value'),
                "article": row.get('article', {}).get('value'),
            }
        pois_dict[uri]["instances"].add(row.get('instanceLabel', {}).get('value', ''))
    pois = list(pois_dict.values())
    n_antes_dedup = len(pois)
    pois = deduplicar_pois(pois)
    n_dedup = n_antes_dedup - len(pois)
    pois.sort(key=score_relevancia)
    con_imagen = sum(1 for p in pois if p['image'])
    con_articulo = sum(1 for p in pois if p['article'])
    con_ambos = sum(1 for p in pois if p['image'] and p['article'])
    dedup_msg = f" (deduplicados: {n_dedup})" if n_dedup else ""
    print(f"  {len(pois)} POIs{dedup_msg} · {con_imagen} con imagen · {con_articulo} con artículo · {con_ambos} con ambos · {res['ms']:.0f} ms · {res['bytes']} B")
    for p in pois:
        img = "📷" if p['image'] else "  "
        art = "📄" if p['article'] else "  "
        inst = ", ".join(sorted(i for i in p['instances'] if i))[:35]
        print(f"    {img}{art} {p['label'][:42]:42s}  ({inst})")
    resultados_wd[m['nombre']] = {
        "ok": True, "ms": res['ms'], "bytes": res['bytes'], "pois": pois,
        "con_imagen": con_imagen, "con_articulo": con_articulo, "con_ambos": con_ambos,
    }
    time.sleep(1)

print("\n--- Resumen prueba 1 ---")
print(f"{'Municipio':32s} {'POIs':>6s} {'Img':>6s} {'Wiki':>6s} {'Img+W':>7s} {'ms':>8s}")
for m in MUNICIPIOS:
    r = resultados_wd.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:32s} {len(r['pois']):>6d} {r['con_imagen']:>6d} {r['con_articulo']:>6d} {r['con_ambos']:>7d} {r['ms']:>8.0f}")
    else:
        print(f"{m['nombre']:32s}   ERROR: {r.get('error','?')[:40]}")

## Prueba 2 — Wikipedia REST API + Wikidata image (cascada multi-fuente)

**Estrategia (versión final tras descubrir que Wikipedia es incompleta para pueblos pequeños):**

Cascada de tres vías por POI:

1. **Vía 1 — Wikipedia REST por URL enlazada:** si Wikidata reporta un artículo en es.wiki vía `schema:about`, pedimos el `summary` directamente.
2. **Vía 2 — Wikipedia REST por construcción de título:** si la vía 1 falla, intentamos construir el título de Wikipedia a partir del label del POI (`Castillo de X` → `Castillo_de_X`).
3. **Vía 3 — Wikidata image (P18) como respaldo de foto:** si Wikipedia no devuelve nada o devuelve una página de desambiguación, usamos la imagen que da Wikidata vía `Special:FilePath?width=400`. La descripción cortita de Wikidata (`schema:description`) sirve como texto de respaldo.

**Detección de páginas de desambiguación:** Wikipedia REST devuelve `type=disambiguation` para páginas como "Iglesia de la Asunción" (que listan todas las del mundo). Estas se descartan explícitamente y se rescatan con la vía 3.

**Aprendizaje:** ninguna API individual es suficiente para cubrir la realidad española de pueblos pequeños. Pedraza es ejemplo: castillo famoso, sin artículo propio en Wikipedia. La cascada cubre 5/5 pueblos en la muestra.

In [ ]:
def titulo_desde_url(article_url):
    """https://es.wikipedia.org/wiki/Castillo_de_Medinaceli -> Castillo_de_Medinaceli"""
    return article_url.rsplit('/wiki/', 1)[-1]

def titulo_desde_label(label):
    """Convierte un label de Wikidata en un título probable de Wikipedia."""
    if not label:
        return None
    titulo = label[0].upper() + label[1:]
    return titulo.replace(' ', '_')

def wiki_summary(titulo):
    url = f"{WIKIPEDIA_REST_ES}/page/summary/{quote(titulo)}"
    return fetch_json(url)

def tamaño_descarga(url):
    """Descarga una URL y devuelve su tamaño en bytes."""
    try:
        r = requests.get(url, headers={"User-Agent": UA}, timeout=15)
        if r.status_code == 200:
            return len(r.content)
    except Exception:
        pass
    return None

def commons_thumb_url(commons_url, ancho=400):
    """Convierte una URL de Wikimedia Commons en una URL de thumbnail al ancho deseado."""
    if not commons_url:
        return None
    sep = '&' if '?' in commons_url else '?'
    return f"{commons_url}{sep}width={ancho}"

def buscar_contenido_para_poi(poi):
    """Cascada Wikipedia → Wikidata image. Devuelve dict con extract, thumb_url, etc."""
    resultado = {
        "extract": None, "thumb_url": None, "thumb_w": None, "thumb_h": None,
        "thumb_bytes": None, "fuentes": [], "ms_total": 0, "json_bytes": 0,
        "es_desambiguacion": False,
    }
    # Vía 1 + 2: Wikipedia
    res = None
    if poi.get('article'):
        titulo = titulo_desde_url(poi['article'])
        res = wiki_summary(titulo)
        if res['ok']:
            resultado['fuentes'].append('wp-link')
    if (not res or not res['ok']) and poi.get('label'):
        titulo = titulo_desde_label(poi['label'])
        res = wiki_summary(titulo)
        if res and res['ok']:
            resultado['fuentes'].append('wp-label')
    if res and res['ok']:
        data = res['data']
        resultado['ms_total'] += res['ms']
        resultado['json_bytes'] += res['bytes']
        if data.get('type') == 'disambiguation':
            resultado['es_desambiguacion'] = True
            resultado['fuentes'].append('DESAMBIGUACION')
        else:
            resultado['extract'] = data.get('extract', '')
            thumb = (data.get('thumbnail') or {}).get('source')
            if thumb:
                resultado['thumb_url'] = thumb
                resultado['thumb_w'] = data['thumbnail'].get('width')
                resultado['thumb_h'] = data['thumbnail'].get('height')
    # Vía 3: imagen de Wikidata como respaldo si no hay thumb de Wikipedia
    if not resultado['thumb_url'] and poi.get('image'):
        thumb_url = commons_thumb_url(poi['image'], ancho=400)
        resultado['thumb_url'] = thumb_url
        resultado['fuentes'].append('wd-image')
    # Texto de respaldo desde la descripción de Wikidata
    if not resultado['extract'] and poi.get('desc'):
        resultado['extract'] = poi['desc']
        resultado['fuentes'].append('wd-desc')
    if resultado['thumb_url']:
        resultado['thumb_bytes'] = tamaño_descarga(resultado['thumb_url'])
    return resultado

resultados_wp = {}

for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    if not r_wd.get('ok'):
        continue
    candidatos = r_wd['pois'][:3]
    print(f"\n=== {nombre} ({len(candidatos)} POIs candidatos) ===")
    resultados_wp[nombre] = []
    for p in candidatos:
        info = buscar_contenido_para_poi(p)
        marca = '+'.join(info['fuentes']) if info['fuentes'] else 'NADA'
        if info['es_desambiguacion']:
            print(f"  ⚠ {p['label'][:42]} [{marca}] descartado por desambiguación")
            if poi_image := p.get('image'):
                info2 = {'thumb_url': commons_thumb_url(poi_image, 400), 'fuentes': ['wd-image-only']}
                info2['thumb_bytes'] = tamaño_descarga(info2['thumb_url'])
                info2['extract'] = p.get('desc', '')
                if info2['extract']:
                    info2['fuentes'].append('wd-desc')
                info = {**info, **info2}
                marca = '+'.join(info['fuentes'])
                print(f"    → rescatado con [{marca}]")
            else:
                continue
        extract_corto = (info.get('extract') or '')[:140].replace('\n', ' ')
        thumb_info = ""
        if info.get('thumb_bytes'):
            thumb_info = f"{info['thumb_bytes']/1024:.1f} KB"
            if info.get('thumb_w'):
                thumb_info = f"{info['thumb_w']}x{info['thumb_h']} · " + thumb_info
        else:
            thumb_info = "(sin foto)"
        print(f"  ✓ {p['label'][:42]} [{marca}]")
        print(f"    foto: {thumb_info}")
        print(f"    texto: {len(info.get('extract') or '')} chars")
        if info.get('extract'):
            print(f"    → {extract_corto}...")
        resultados_wp[nombre].append({
            "poi": p['label'], "fuentes": info['fuentes'],
            "extract": info.get('extract'), "thumb_bytes": info.get('thumb_bytes'),
            "thumb_url": info.get('thumb_url'), "json_bytes": info['json_bytes'],
        })
        time.sleep(0.3)

print("\n--- Resumen prueba 2 ---")
print("Tamaño estimado vista 3 (1 foto principal + jsons por pueblo)")
total_kb = 0
for nombre, items in resultados_wp.items():
    if not items:
        print(f"  {nombre:32s}  (sin contenido accesible)")
        continue
    items_con_thumb = [i for i in items if i['thumb_bytes']]
    kb_foto = (items_con_thumb[0]['thumb_bytes'] / 1024) if items_con_thumb else 0
    kb_json = sum(i['json_bytes'] for i in items) / 1024
    subtotal = kb_foto + kb_json
    total_kb += subtotal
    n_con_texto = sum(1 for i in items if i['extract'])
    n_con_foto = len(items_con_thumb)
    print(f"  {nombre:32s}  {n_con_foto}/3 fotos · {n_con_texto}/3 textos · {subtotal:5.1f} KB")
print(f"  {'TOTAL vista 3':32s}  {total_kb:5.1f} KB")

## Prueba 3 — Overpass como fallback geográfico (PENDIENTE)

**Estado al cierre de la sesión 05: NO COMPLETADA.** Los tres mirrors públicos de Overpass (overpass-api.de, overpass.kumi.systems, overpass.private.coffee) estaban saturados o caídos durante toda la sesión. Aplazada a una sesión específica de Overpass.

**Estrategia previstas para cuando se pueda ejecutar:** para los municipios donde Wikidata haya dado pocos POIs (menos de 3, por ejemplo), consultamos Overpass en un radio de 2 km alrededor del centro del pueblo con los tags `historic=*`, `tourism=viewpoint`, `tourism=attraction`, `natural=peak`, `building=castle|church|monastery|cathedral|chapel`.

**Qué se medirá:**
- Número de elementos devueltos.
- Cuántos tienen `name` (los que no, no nos sirven).
- Latencia (Overpass puede ser lento).
- Tamaño de la respuesta.
- Si Overpass aporta POIs significativos que no estaban en Wikidata.

In [ ]:
def overpass_pois_cercanos(lat, lon, radio_m=2000):
    """POIs históricos, turísticos y naturales en un radio alrededor de un punto."""
    q = f'''[out:json][timeout:50];
(
  node["historic"](around:{radio_m},{lat},{lon});
  way["historic"](around:{radio_m},{lat},{lon});
  node["tourism"="viewpoint"](around:{radio_m},{lat},{lon});
  node["tourism"="attraction"](around:{radio_m},{lat},{lon});
  way["tourism"="attraction"](around:{radio_m},{lat},{lon});
  node["natural"="peak"](around:{radio_m},{lat},{lon});
  way["building"~"^(castle|church|monastery|cathedral|chapel)$"](around:{radio_m},{lat},{lon});
);
out center tags;
'''
    return fetch_overpass(q)

resultados_ov = {}

for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    n_wd = len(r_wd.get('pois', []))
    print(f"\n=== {nombre} (Wikidata dio {n_wd} POIs, lanzamos Overpass de todas formas) ===")
    res = overpass_pois_cercanos(m['lat'], m['lon'], radio_m=2000)
    if not res['ok']:
        print(f"  ERROR: {res['error'][:150]}")
        resultados_ov[nombre] = {"ok": False, "error": res['error']}
        continue
    elementos = res['data'].get('elements', [])
    con_nombre = [e for e in elementos if e.get('tags', {}).get('name')]
    print(f"  {len(elementos)} elementos · {len(con_nombre)} con name · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
    for e in con_nombre[:15]:
        tags = e.get('tags', {})
        name = tags.get('name', '?')
        tipo_info = []
        for k in ('historic', 'tourism', 'natural', 'building'):
            if k in tags:
                tipo_info.append(f"{k}={tags[k]}")
        tipo = ' '.join(tipo_info)
        print(f"    {name[:42]:42s}  [{tipo}]")
    if len(con_nombre) > 15:
        print(f"    ... y {len(con_nombre)-15} más")
    resultados_ov[nombre] = {
        "ok": True, "ms": res['ms'], "bytes": res['bytes'],
        "total": len(elementos), "con_nombre": len(con_nombre), "elementos": con_nombre,
    }
    time.sleep(3)  # cortesía con Overpass: pausa generosa entre municipios

print("\n--- Comparativa Wikidata vs Overpass ---")
print(f"{'Municipio':32s} {'WD':>6s} {'WD+img':>8s} {'OV':>6s} {'OV+nom':>8s}")
for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    r_ov = resultados_ov.get(nombre, {})
    n_wd = len(r_wd.get('pois', [])) if r_wd.get('ok') else -1
    n_wd_img = r_wd.get('con_imagen', 0) if r_wd.get('ok') else 0
    n_ov = r_ov.get('total', 0) if r_ov.get('ok') else -1
    n_ov_n = r_ov.get('con_nombre', 0) if r_ov.get('ok') else 0
    estado_ov = "ERROR" if not r_ov.get('ok') else f"{n_ov:>6d}"
    print(f"{nombre:32s} {n_wd:>6d} {n_wd_img:>8d} {estado_ov} {n_ov_n:>8d}")

## Prueba 4 — Datos del municipio (cierra el bloque 1.5 del plan)

**Estado: COMPLETADA en sesión 05. Cubre completamente el bloque 1.5 del plan original.**

**Estrategia:** una segunda consulta SPARQL por municipio pidiendo: población (P1082), altitud (P2044), superficie (P2046), part of (P361, para comarca/mancomunidad). Una sola consulta por municipio, latencia muy baja.

**Resultado de la sesión 05:** 5/5 municipios devuelven los tres primeros campos, 4/5 el cuarto (Pedraza no tiene comarca formal). Latencias 130-315 ms. Tamaño <1 KB.

**Limitación conocida:** el campo P361 devuelve mancomunidades en lugar de comarcas geográficas (Mancomunidad Comarca de Trujillo, Los Pueblos Más Bonitos de España). Para el panel real habrá que filtrar por instancia 'comarca de España' o aceptar que el campo se omita cuando no haya un valor útil. Va al log de la sesión 05 como mejora pendiente.

In [ ]:
def query_municipio(qid):
    return f'''
SELECT ?poblacion ?altitud ?superficie ?comarcaLabel ?articulo WHERE {{
  OPTIONAL {{ wd:{qid} wdt:P1082 ?poblacion. }}
  OPTIONAL {{ wd:{qid} wdt:P2044 ?altitud. }}
  OPTIONAL {{ wd:{qid} wdt:P2046 ?superficie. }}
  OPTIONAL {{ wd:{qid} wdt:P361 ?comarca. }}
  OPTIONAL {{
    ?articulo schema:about wd:{qid} ;
              schema:isPartOf <https://es.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es,en". }}
}}
LIMIT 1
'''

resultados_muni = {}

for m in MUNICIPIOS:
    q = query_municipio(m['qid'])
    res = consultar_wikidata(q)
    if not res['ok']:
        print(f"{m['nombre']:32s}  ERROR: {res['error'][:80]}")
        continue
    rows = res['data']['results']['bindings']
    if not rows:
        print(f"{m['nombre']:32s}  (sin datos)")
        continue
    row = rows[0]
    pob = row.get('poblacion', {}).get('value', '-')
    alt = row.get('altitud', {}).get('value', '-')
    sup = row.get('superficie', {}).get('value', '-')
    com = row.get('comarcaLabel', {}).get('value', '-')
    art = row.get('articulo', {}).get('value', '-')
    print(f"{m['nombre']:32s}  pob={pob:>8s}  alt={alt:>6s}  sup={sup:>6s}  com={com[:25]:25s}  ({res['ms']:.0f} ms)")
    resultados_muni[m['nombre']] = {
        "poblacion": pob, "altitud": alt, "superficie": sup, "comarca": com, "articulo": art, "ms": res['ms']
    }
    time.sleep(1)

print("\nCampos disponibles por municipio:")
for nombre, d in resultados_muni.items():
    campos = [k for k in ('poblacion','altitud','superficie','comarca') if d.get(k) and d[k] != '-']
    print(f"  {nombre:32s} {len(campos)}/4  ({', '.join(campos)})")

## Prueba 5 — Pueblos cercanos a un punto GPS (PENDIENTE)

**Estado al cierre de la sesión 05: NO COMPLETADA por la misma razón que la prueba 3** (saturación de los mirrors de Overpass).

**Por qué es crítica:** la vista 3 del panel muestra 4 pueblos cercanos al punto GPS actual. Sin esta consulta, no hay vista 3.

**Estrategia prevista:** Overpass con `place=village|town|city` en un radio de 15 km, ordenados luego en cliente por distancia al punto.

**Puntos de prueba:** 3 puntos de carretera real (no sobre un pueblo) para simular el caso del coche circulando: A-2 cerca de Medinaceli, A-5 cerca de Trujillo, N-VI cerca de Astorga.

In [ ]:
import math

def distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def overpass_pueblos_cercanos(lat, lon, radio_m=15000):
    q = f'''[out:json][timeout:50];
(
  node["place"~"^(village|town|city)$"](around:{radio_m},{lat},{lon});
);
out tags;
'''
    return fetch_overpass(q)

PUNTOS_CARRETERA = [
    {"nombre": "A-2 cerca de Medinaceli", "lat": 41.1856, "lon": -2.4570},
    {"nombre": "A-5 cerca de Trujillo",   "lat": 39.4760, "lon": -5.8200},
    {"nombre": "N-VI cerca de Astorga",   "lat": 42.4420, "lon": -6.0800},
]

resultados_cercanos = {}

for p in PUNTOS_CARRETERA:
    print(f"\n=== {p['nombre']} ({p['lat']}, {p['lon']}) ===")
    res = overpass_pueblos_cercanos(p['lat'], p['lon'], radio_m=15000)
    if not res['ok']:
        print(f"  ERROR: {res['error'][:150]}")
        resultados_cercanos[p['nombre']] = {"ok": False, "error": res['error']}
        continue
    elementos = res['data'].get('elements', [])
    pueblos = []
    for e in elementos:
        tags = e.get('tags', {})
        name = tags.get('name')
        if not name:
            continue
        d = distancia_m(p['lat'], p['lon'], e.get('lat', 0), e.get('lon', 0))
        pueblos.append({
            "name": name, "place": tags.get('place'),
            "dist": d, "pop": tags.get('population'),
        })
    pueblos.sort(key=lambda x: x['dist'])
    print(f"  {len(pueblos)} pueblos con nombre en radio 15 km · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
    print("  Primeros 10 por distancia:")
    for pu in pueblos[:10]:
        pop = f"pop={pu['pop']}" if pu['pop'] else ""
        print(f"    {pu['dist']/1000:5.1f} km  {pu['name'][:32]:32s} [{pu['place']}] {pop}")
    resultados_cercanos[p['nombre']] = {
        "ok": True, "ms": res['ms'], "bytes": res['bytes'],
        "total": len(pueblos), "top4": pueblos[:4],
    }
    time.sleep(3)

## Resumen final

Celda que reúne todo en un informe compacto. Si las pruebas 3 y 5 no se ejecutaron (Overpass saturado), simplemente saldrán vacías o con errores.

In [ ]:
print("="*70)
print("RESUMEN FASE 1.4 — POIs por pueblo")
print("="*70)

print("\n## Wikidata SPARQL (prueba 1)")
print(f"{'Municipio':32s} {'POIs':>6s} {'Img':>6s} {'Wiki':>6s} {'Img+W':>7s} {'ms':>8s}")
for m in MUNICIPIOS:
    r = resultados_wd.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:32s} {len(r['pois']):>6d} {r['con_imagen']:>6d} {r['con_articulo']:>6d} {r['con_ambos']:>7d} {r['ms']:>8.0f}")

print("\n## Wikipedia REST + Wikidata image (prueba 2)")
total = 0
for nombre, items in resultados_wp.items():
    if not items: continue
    items_con_thumb = [i for i in items if i['thumb_bytes']]
    kb_foto = (items_con_thumb[0]['thumb_bytes'] / 1024) if items_con_thumb else 0
    kb_json = sum(i['json_bytes'] for i in items) / 1024
    n_con_texto = sum(1 for i in items if i['extract'])
    n_con_foto = len(items_con_thumb)
    total += kb_foto + kb_json
    print(f"  {nombre:32s}  {n_con_foto}/3 fotos · {n_con_texto}/3 textos · {kb_foto+kb_json:5.1f} KB")
print(f"  TOTAL estimado vista 3: {total:.1f} KB")

print("\n## Overpass fallback POIs (prueba 3)")
if not resultados_ov:
    print("  (no ejecutada o sin resultados)")
else:
    print(f"{'Municipio':32s} {'elem':>6s} {'con nom':>8s} {'ms':>8s}")
    for m in MUNICIPIOS:
        r = resultados_ov.get(m['nombre'], {})
        if r.get('ok'):
            print(f"{m['nombre']:32s} {r['total']:>6d} {r['con_nombre']:>8d} {r['ms']:>8.0f}")
        elif r:
            print(f"{m['nombre']:32s}   ERROR")

print("\n## Datos de municipio / bloque 1.5 (prueba 4)")
for nombre, d in resultados_muni.items():
    campos = [k for k in ('poblacion','altitud','superficie','comarca') if d.get(k) and d[k] != '-']
    print(f"  {nombre:32s} {len(campos)}/4 campos disponibles: {', '.join(campos)}")

print("\n## Pueblos cercanos vía Overpass (prueba 5)")
if not resultados_cercanos:
    print("  (no ejecutada o sin resultados)")
else:
    for nombre, r in resultados_cercanos.items():
        if r.get('ok'):
            print(f"  {nombre:32s} {r['total']:3d} pueblos en 15 km · {r['ms']:.0f} ms")
            for pu in r['top4']:
                print(f"    {pu['dist']/1000:5.1f} km  {pu['name']}")
        else:
            print(f"  {nombre:32s} ERROR")

print("\n" + "="*70)
print("Pega este resumen en el chat para que Claude lo analice")
print("y mantenga actualizado el documento de seguimiento.")